# Spanner Data Boost Streaming Demo

![DataBoost Streaming Demo](SpannerStreamingLakehouse-Dataflow.drawio.png)

This notebook is the single source of truth for the demo: it drives GCP
resource provisioning (via `gcloud`, executed as shell cells), builds and
submits the Spark streaming jobs, runs the producer, provisions the BigQuery
reservation and scheduled query, starts the continuous queries, and contains
the dashboard and ad-hoc query code used for analysis.

**Two execution contexts:**

- **Local cells** (marked `%%bash`) run on your workstation with `gcloud`
  configured, provisioning infrastructure and submitting jobs.
- **Cluster-side cells** (Part 13) run inside a JupyterLab **Python 3** kernel
  launched from the Managed Spark cluster's Web Interfaces, because they need
  private-network access to Spanner/Iceberg that only exists inside the VPC.
  Copy those cells into a notebook there (or open this same file in that
  JupyterLab instance) and run them with a Python 3 kernel.

## Architecture

```text
Custom VPC: spark-databoost-demo-vpc
Subnet: 10.10.0.0/24
  |
  +-- Private Google Access
  |
  +-- Cloud Router + Cloud NAT
  |     Outbound access for Maven packages and Docker images
  |
  +-- IAP
  |     SSH access to the Kafka VM without a public IP
  |
  +-- Private Kafka VM
  |     +-- Container-Optimized OS
  |     +-- apache/kafka-native:4.1.2
  |     +-- tpch-generator:local (built from source src/java-tpch-stream-generator)
  |
  +-- Private single-node Managed Spark (Dataproc) cluster
        +-- Jupyter optional component
        +-- Kafka, Spanner, and Iceberg dependencies
        +-- Kafka-to-Lakehouse streaming job
        +-- Kafka-to-Spanner streaming job

Cloud Storage:
  +-- Job bucket (staging and checkpoints)
  +-- Warehouse bucket (Iceberg data and metadata)

Lakehouse aggregate path:
  +-- BigQuery scheduled query every 5 minutes
  |     +-- Identify aggregate keys affected in the last 5 minutes
  |     +-- Recompute cumulative COUNT and SUM snapshots for those keys
  |     +-- Append snapshots to three native BigQuery staging tables
  +-- Three BigQuery continuous queries
        +-- Consume staging-table appends with APPENDS()
        +-- Calculate averages with SAFE_DIVIDE
        +-- Upsert snapshots into three Spanner aggregate tables

Spanner:
  +-- Raw orders populated directly from Kafka by Spark
  +-- Three aggregate tables populated through the lakehouse path
```


## 1. Prerequisites and variables

Before starting:

- Run `gcloud auth login`
- Choose a region that supports all services used by the demo; `ZONE` must belong to `REGION`.

The user running setup needs these roles:

| Category | Role name and ID | Purpose |
| :--- | :--- | :--- |
| **IAM and policy** | **Project IAM Admin**<br>`roles/resourcemanager.projectIamAdmin` | Grant project-level IAM roles to the Spark service account |
| | **Service Account Admin**<br>`roles/iam.serviceAccountAdmin` | Create, manage, and delete the service account |
| | **Service Account User**<br>`roles/iam.serviceAccountUser` | Attach and impersonate the service account for demo jobs |
| **Network and compute** | **Compute Network Admin**<br>`roles/compute.networkAdmin` | Create and delete the VPC, subnet, firewall rules, router, and NAT |
| | **Compute Instance Admin**<br>`roles/compute.instanceAdmin.v1` | Create, manage, and delete the Kafka VM |
| | **IAP-secured Tunnel User**<br>`roles/iap.tunnelResourceAccessor` | Use SSH and SCP through IAP to reach the private VM |
| **Data and analytics** | **Cloud Spanner Admin**<br>`roles/spanner.admin` | Create the Spanner instance, database, schema, and execute SQL |
| | **Dataproc Admin**<br>`roles/dataproc.admin` | Create the cluster, submit jobs, access JupyterLab, and delete the cluster |
| | **BigLake Admin**<br>`roles/biglake.admin` | Create and delete the Iceberg REST catalog |
| | **BigQuery Admin**<br>`roles/bigquery.admin` | Create datasets, tables, reservations, continuous queries, and scheduled queries |
| **Storage and services** | **Storage Admin**<br>`roles/storage.admin` | Create and delete buckets and grant bucket IAM |
| | **Service Usage Admin**<br>`roles/serviceusage.serviceUsageAdmin` | Enable required APIs |

Edit the values in the next cell. Running it writes a local `.env` file used by later shell, SCP, and SSH cells.

**Everything has a default that works EXCEPT the PROJECT_ID that mandatory must be edited with your project name.**

> **Warning:** `%%writefile` overwrites an existing `.env`.

In [ ]:
%%writefile .env
# !! MANDATORY !! Edit PROJECT_ID
export PROJECT_ID="your-project-id"
export REGION="us-central1"
export ZONE="us-central1-a"

# Network
export NETWORK="spark-databoost-demo-vpc"
export SUBNET="spark-databoost-demo-subnet"
export SUBNET_CIDR="10.10.0.0/24"
export ROUTER="${NETWORK}-router"
export NAT="${NETWORK}-nat"

# Demo resources
export DEMO_ID="spark-databoost-demo"
export KAFKA_VM="${DEMO_ID}-kafka"
export CLUSTER="${DEMO_ID}-cluster"
export SA_NAME="${DEMO_ID}-sa"
export SPARK_SA="${SA_NAME}@${PROJECT_ID}.iam.gserviceaccount.com"

# Storage
export JOB_BUCKET="${PROJECT_ID}-${DEMO_ID}-jobs"
export WAREHOUSE_BUCKET="${PROJECT_ID}-${DEMO_ID}-warehouse"

# Lakehouse
export LAKEHOUSE_CATALOG="${WAREHOUSE_BUCKET}"
export LAKEHOUSE_NAMESPACE="demo"

# Spanner
export SPANNER_INSTANCE="${DEMO_ID}-instance"
export SPANNER_DATABASE="demo"

# Kafka
export KAFKA_TOPIC="orders"
export KAFKA_BROKER="${KAFKA_VM}.${ZONE}.c.${PROJECT_ID}.internal:9092"

# BigQuery
export BQ_DATASET="spark_databoost_demo_lakehouse_ds"
export BQ_RESERVATION="${DEMO_ID}-bq-reservation"

Load the variables and configure `gcloud`:

In [ ]:
%%bash
source .env
set -euo pipefail

gcloud config set project "$PROJECT_ID"
gcloud config set dataproc/region "$REGION"


## 2. Enable APIs

In [ ]:
%%bash
source .env

gcloud services enable \
  compute.googleapis.com \
  dataproc.googleapis.com \
  iam.googleapis.com \
  biglake.googleapis.com \
  bigquery.googleapis.com \
  bigqueryreservation.googleapis.com \
  bigquerydatatransfer.googleapis.com \
  storage.googleapis.com \
  spanner.googleapis.com \
  serviceusage.googleapis.com


## 3. Create private network, subnet, NAT, and firewall rules

### 3.1 Create the VPC and subnet


In [ ]:
%%bash
source .env
set -euo pipefail

if gcloud compute networks describe "$NETWORK" &>/dev/null; then
  echo "Network $NETWORK already exists, skipping."
else
  gcloud compute networks create "$NETWORK" \
    --subnet-mode=custom
fi

if gcloud compute networks subnets describe "$SUBNET" --region="$REGION" &>/dev/null; then
  echo "Subnet $SUBNET already exists, skipping."
else
  gcloud compute networks subnets create "$SUBNET" \
    --network="$NETWORK" \
    --region="$REGION" \
    --range="$SUBNET_CIDR" \
    --enable-private-ip-google-access
fi


### 3.2 Allow internal subnet communication

In [ ]:
%%bash
source .env
set -euo pipefail

if gcloud compute firewall-rules describe "allow-${NETWORK}-internal" &>/dev/null; then
  echo "Firewall rule allow-${NETWORK}-internal already exists, skipping."
else
  gcloud compute firewall-rules create "allow-${NETWORK}-internal" \
    --network="$NETWORK" \
    --direction=INGRESS \
    --action=ALLOW \
    --rules=tcp,udp,icmp \
    --source-ranges="$SUBNET_CIDR"
fi


### 3.3 Allow IAP SSH access

In [ ]:
%%bash
source .env
set -euo pipefail

if gcloud compute firewall-rules describe "allow-${NETWORK}-ssh-iap" &>/dev/null; then
  echo "Firewall rule allow-${NETWORK}-ssh-iap already exists, skipping."
else
  gcloud compute firewall-rules create "allow-${NETWORK}-ssh-iap" \
    --network="$NETWORK" \
    --direction=INGRESS \
    --action=ALLOW \
    --rules=tcp:22 \
    --source-ranges="35.235.240.0/20"
fi


### 3.4 Create Cloud Router and Cloud NAT

In [ ]:
%%bash
source .env
set -euo pipefail

if gcloud compute routers describe "$ROUTER" --region="$REGION" &>/dev/null; then
  echo "Router $ROUTER already exists, skipping."
else
  gcloud compute routers create "$ROUTER" \
    --network="$NETWORK" \
    --region="$REGION"
fi

if gcloud compute routers nats describe "$NAT" --router="$ROUTER" --region="$REGION" &>/dev/null; then
  echo "NAT $NAT already exists, skipping."
else
  gcloud compute routers nats create "$NAT" \
    --router="$ROUTER" \
    --region="$REGION" \
    --auto-allocate-nat-external-ips \
    --nat-custom-subnet-ip-ranges="$SUBNET"
fi


## 4. Create Service Account and IAM grants

Create the demo service account:


In [ ]:
%%bash
source .env
set -euo pipefail

if gcloud iam service-accounts describe "$SPARK_SA" &>/dev/null; then
  echo "Service account $SPARK_SA already exists, skipping."
else
  gcloud iam service-accounts create "$SA_NAME" \
    --display-name="Spark Data Boost demo"
fi


Grant the required project-level roles

In [ ]:
%%bash
source .env
set -euo pipefail

for ROLE in \
  roles/dataproc.worker \
  roles/spanner.databaseUser \
  roles/spanner.viewer \
  roles/spanner.databaseReaderWithDataBoost \
  roles/biglake.editor \
  roles/bigquery.jobUser \
  roles/bigquery.dataEditor \
  roles/serviceusage.serviceUsageConsumer \
  roles/logging.logWriter; do
  echo "Granting $ROLE to $SPARK_SA..."
  gcloud projects add-iam-policy-binding "$PROJECT_ID" \
    --member="serviceAccount:${SPARK_SA}" \
    --role="$ROLE" \
    --condition=None \
    --quiet > /dev/null
done
echo "Done."

Allow the current user to attach the service account:

In [ ]:
%%bash
source .env
set -euo pipefail

export CURRENT_USER="$(gcloud config get-value account)"

gcloud iam service-accounts add-iam-policy-binding "$SPARK_SA" \
  --member="user:${CURRENT_USER}" \
  --role="roles/iam.serviceAccountUser"


Grant the BigQuery Data Transfer Service agent permission to impersonate `$SPARK_SA`.

In [ ]:
%%bash
source .env
set -euo pipefail

PROJECT_NUMBER=$(gcloud projects describe "$PROJECT_ID" --format='value(projectNumber)')
DTS_SA="service-${PROJECT_NUMBER}@gcp-sa-bigquerydatatransfer.iam.gserviceaccount.com"

echo "Granting Token Creator on $SPARK_SA to $DTS_SA..."
gcloud iam service-accounts add-iam-policy-binding "$SPARK_SA" \
  --member="serviceAccount:${DTS_SA}" \
  --role="roles/iam.serviceAccountTokenCreator" \
  --condition=None \
  --quiet >/dev/null

## 5. Create job/checkpoint and warehouse buckets

### 5.1 Create the job bucket

The job bucket stores Dataproc staging files, PySpark applications, and streaming checkpoints.


In [ ]:
%%bash
source .env
set -euo pipefail

if gcloud storage buckets describe "gs://${JOB_BUCKET}" &>/dev/null; then
  echo "Bucket gs://${JOB_BUCKET} already exists, skipping creation."
else
  gcloud storage buckets create "gs://${JOB_BUCKET}" \
    --location="$REGION" \
    --uniform-bucket-level-access
fi

gcloud storage buckets add-iam-policy-binding "gs://${JOB_BUCKET}" \
  --member="serviceAccount:${SPARK_SA}" \
  --role="roles/storage.objectAdmin"


### 5.2 Create the warehouse bucket

The warehouse bucket stores Iceberg table data and metadata.


In [ ]:
%%bash
source .env
set -euo pipefail

if gcloud storage buckets describe "gs://${WAREHOUSE_BUCKET}" &>/dev/null; then
  echo "Bucket gs://${WAREHOUSE_BUCKET} already exists, skipping."
else
  gcloud storage buckets create "gs://${WAREHOUSE_BUCKET}" \
    --location="$REGION" \
    --uniform-bucket-level-access
fi


## 6. Create the Lakehouse Iceberg REST catalog

In the Google Cloud console:

1. Open **BigQuery / Lakehouse**.
2. Select **Create catalog**.
3. Select **Iceberg REST catalog**.
4. Select **Single bucket catalog**.
5. Select `gs://${WAREHOUSE_BUCKET}` as the Cloud Storage bucket.
6. Select **Credential vending** as the authentication method.
7. Create the catalog.
8. Select **Set bucket permissions**.

The final action gives the auto-provisioned Lakehouse catalog service account access to the warehouse bucket. Spark receives short-lived, scoped storage credentials through the REST catalog.


## 7. Create Kafka VM and start Kafka

### 7.1 Create the private Kafka VM


In [ ]:
%%bash
source .env
set -euo pipefail

if gcloud compute instances describe "$KAFKA_VM" --zone="$ZONE" &>/dev/null; then
  echo "Instance $KAFKA_VM already exists, skipping."
else
  gcloud compute instances create "$KAFKA_VM" \
    --zone="$ZONE" \
    --machine-type="e2-small" \
    --network="$NETWORK" \
    --subnet="$SUBNET" \
    --no-address \
    --image-family="cos-stable" \
    --image-project="cos-cloud" \
    --boot-disk-size="10GB" \
    --boot-disk-type="pd-standard"
fi


### 7.2 Copy the source files to the VM

In [ ]:
%%bash
source .env
set -euo pipefail

gcloud compute scp \
  --recurse \
  src \
  .env \
  "${KAFKA_VM}:~/" \
  --zone="$ZONE" \
  --tunnel-through-iap


### 7.3 Start Kafka and build the producer image

This step starts the Kafka broker and builds the data generator Docker image from source. 
Expect this cell to take 3 to 5 minutes to complete.

In [ ]:
%%bash
source .env
set -euo pipefail

gcloud compute ssh "$KAFKA_VM" \
  --zone="$ZONE" \
  --tunnel-through-iap \
  --command='
    set -euo pipefail

    source ~/.env

    : "${KAFKA_BROKER:?KAFKA_BROKER is not defined in ~/.env}"
    : "${KAFKA_TOPIC:?KAFKA_TOPIC is not defined in ~/.env}"

    sudo iptables -C INPUT -p tcp --dport 9092 -j ACCEPT \
      2>/dev/null || \
      sudo iptables -A INPUT -p tcp --dport 9092 -j ACCEPT

    docker rm -f kafka 2>/dev/null || true

    docker run -d \
      --name kafka \
      --network host \
      -e KAFKA_NODE_ID=1 \
      -e KAFKA_PROCESS_ROLES=broker,controller \
      -e KAFKA_CONTROLLER_QUORUM_VOTERS=1@localhost:9093 \
      -e KAFKA_LISTENERS=PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093 \
      -e KAFKA_ADVERTISED_LISTENERS=PLAINTEXT://${KAFKA_BROKER} \
      -e KAFKA_CONTROLLER_LISTENER_NAMES=CONTROLLER \
      -e KAFKA_LISTENER_SECURITY_PROTOCOL_MAP=CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT \
      -e KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR=1 \
      -e KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR=1 \
      -e KAFKA_TRANSACTION_STATE_LOG_MIN_ISR=1 \
      -e KAFKA_DEFAULT_REPLICATION_FACTOR=1 \
      apache/kafka-native:4.1.2

    echo "Waiting for Kafka and creating topic: $KAFKA_TOPIC"

    for attempt in $(seq 1 10); do
      echo "Attempt $attempt of 10 in 30 seconds..."
      sleep 30

      if docker run --rm \
        --network host \
        apache/kafka:4.1.2 \
        /opt/kafka/bin/kafka-topics.sh \
          --bootstrap-server localhost:9092 \
          --create \
          --if-not-exists \
          --topic "$KAFKA_TOPIC" \
          --partitions 1 \
          --replication-factor 1; then
        echo "Topic is ready: $KAFKA_TOPIC"
        break
      fi
      if [ "$attempt" -eq 10 ]; then
        echo "Failed to create topic after 10 attempts" >&2
        docker logs kafka >&2
        exit 1
      fi
    done

    cd ~/src/java-tpch-stream-generator
    docker build --tag tpch-generator:local .
  '


## 8. Create Spanner instance and schema

### 8.1 Create the Spanner instance


In [ ]:
%%bash
source .env
set -euo pipefail

if gcloud spanner instances describe "$SPANNER_INSTANCE" &>/dev/null; then
  echo "Spanner instance $SPANNER_INSTANCE already exists, skipping."
else
  gcloud spanner instances create "$SPANNER_INSTANCE" \
    --config="regional-${REGION}" \
    --description="Spanner Data Boost demo" \
    --edition="ENTERPRISE" \
    --processing-units=100
fi


### 8.2 Create the database and tables

In [ ]:
%%bash
source .env
set -euo pipefail

if gcloud spanner databases describe "$SPANNER_DATABASE" --instance="$SPANNER_INSTANCE" &>/dev/null; then
  echo "Spanner database $SPANNER_DATABASE already exists."
else
  gcloud spanner databases create "$SPANNER_DATABASE" \
    --instance="$SPANNER_INSTANCE"
fi

gcloud spanner databases ddl update "$SPANNER_DATABASE" \
  --instance="$SPANNER_INSTANCE" \
  --ddl="CREATE TABLE IF NOT EXISTS orders (
    order_key INT64 NOT NULL,
    customer_key INT64,
    order_status STRING(1) NOT NULL,
    total_price NUMERIC,
    order_date DATE NOT NULL,
    order_priority STRING(15),
    clerk STRING(15) NOT NULL,
    ship_priority INT64,
    comment STRING(79),
    kafka_timestamp TIMESTAMP,
    ingested_time TIMESTAMP
  ) PRIMARY KEY (order_key);

  CREATE TABLE IF NOT EXISTS order_daily_aggregate (
    order_date DATE NOT NULL,
    order_count INT64 NOT NULL,
    order_total NUMERIC NOT NULL,
    order_total_average NUMERIC NOT NULL,
    last_updated TIMESTAMP
  ) PRIMARY KEY (order_date);

  CREATE TABLE IF NOT EXISTS clerk_status_daily_aggregate (
    clerk STRING(15) NOT NULL,
    order_status STRING(1) NOT NULL,
    order_date DATE NOT NULL,
    orders_processed INT64 NOT NULL,
    total_revenue NUMERIC NOT NULL,
    avg_order_value NUMERIC NOT NULL,
    last_updated TIMESTAMP
  ) PRIMARY KEY (clerk, order_status, order_date);

  CREATE TABLE IF NOT EXISTS order_status_daily_aggregate (
    order_status STRING(1) NOT NULL,
    order_date DATE NOT NULL,
    orders_processed INT64 NOT NULL,
    total_revenue NUMERIC NOT NULL,
    avg_order_value NUMERIC NOT NULL,
    last_updated TIMESTAMP
  ) PRIMARY KEY (order_status, order_date);"

### 8.3 Enable columnar storage for Orders table

In [ ]:
%%bash
source .env
set -euo pipefail

gcloud spanner databases ddl update $SPANNER_DATABASE \
  --instance=$SPANNER_INSTANCE \
  --ddl="ALTER TABLE orders SET OPTIONS (columnar_policy = 'enabled');"


## 9. Create a single-node Managed Spark cluster


In [ ]:
%%bash
source .env
set -euo pipefail

MANAGED_SPARK_IMAGE="2.3-debian12"

if gcloud dataproc clusters describe "$CLUSTER" --region="$REGION" &>/dev/null; then
  echo "Dataproc cluster $CLUSTER already exists, skipping."
else
  gcloud dataproc clusters create "$CLUSTER" \
    --region="$REGION" \
    --zone="$ZONE" \
    --subnet="$SUBNET" \
    --no-address \
    --single-node \
    --image-version="$MANAGED_SPARK_IMAGE" \
    --master-machine-type=n2d-standard-8 \
    --master-boot-disk-type=pd-ssd \
    --master-boot-disk-size="100GB" \
    --bucket="$JOB_BUCKET" \
    --temp-bucket="$JOB_BUCKET" \
    --service-account="$SPARK_SA" \
    --optional-components=JUPYTER \
    --enable-component-gateway \
    --scopes="cloud-platform"
fi


Expect the cluster to be created in a few minutes.

## 10. Upload and submit both streaming jobs

**Streaming flow:** The 2 Spark jobs process data in micro-batches governed by:

- **processing-time:** The fixed clock interval between micro-batch triggers. When each interval arrives, Spark immediately polls Kafka and processes whatever records are currently available at that exact moment.

- **max-offsets-per-trigger:** A hard upper limit (ceiling) on the number of Kafka records pulled in a single micro-batch. If Kafka has a massive backlog, Spark caps the batch size at this number to prevent memory overload.

### 10.1 Write the PySpark application sources

The two streaming job sources live in these cells so the code and its
submission command stay together. Running the cell writes the file to
`src/` for upload.


In [ ]:
%%writefile src/kafka_to_lakehouse.py
#!/usr/bin/env python3

# Requires Jars in Spark env.
#   - spark-iceberg connector
#   - spark-sql-kafka connector
import argparse
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import col, split, to_date, current_timestamp
from pyspark.sql.types import DecimalType, LongType, IntegerType


def parse_args():
    """Parses command-line arguments using argparse."""
    parser = argparse.ArgumentParser(
        description="Stream data from Kafka into an Apache Iceberg lakehouse table."
    )

    parser.add_argument("--bootstrap", required=True, help="Kafka bootstrap servers connection string")
    parser.add_argument("--kafka-topic", required=True, help="Kafka topic name to subscribe to")
    parser.add_argument("--lakehouse-catalog", required=True, help="Target Iceberg catalog name")
    parser.add_argument("--lakehouse-namespace", required=True, help="Target Iceberg namespace/database")
    parser.add_argument("--lakehouse-table", required=True, help="Target Iceberg table name")
    parser.add_argument("--spark-stream-checkpoint", required=True, help="Remote or local for streaming checkpoints")
    parser.add_argument(
        "--processing-time",
        default="30 seconds",
        help="Processing time interval for stream trigger (e.g., '10 seconds', '1 minute', '0 seconds'). Default: '30 seconds'"
    )
    parser.add_argument(
        "--max-offsets-per-trigger",
        default="100",
        help="Maximum number of Kafka offsets processed per trigger cycle. Default: 100"
    )

    return parser.parse_args()


def read_orders(spark: SparkSession, bootstrap_servers: str, topic: str, max_offsets: str) -> DataFrame:
    reader = (
        spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", bootstrap_servers)
        .option("subscribe", topic)
        .option("startingOffsets", "earliest")
        .option("maxOffsetsPerTrigger", max_offsets)
        .option("failOnDataLoss", "true")
    )

    # Extract raw value and Kafka record timestamp
    payload = reader.load().select(
        col("value").cast("string").alias("value"),
        col("timestamp").alias("kafka_timestamp")
    )
    fields = split(col("value"), r"\|")
    return payload.select(
        fields.getItem(0).cast(LongType()).alias("order_key"),
        fields.getItem(1).cast(LongType()).alias("customer_key"),
        fields.getItem(2).alias("order_status"),
        fields.getItem(3).cast(DecimalType(15, 2)).alias("total_price"),
        to_date(fields.getItem(4), "yyyy-MM-dd").alias("order_date"),
        fields.getItem(5).alias("order_priority"),
        fields.getItem(6).alias("clerk"),
        fields.getItem(7).cast(IntegerType()).alias("ship_priority"),
        fields.getItem(8).alias("comment"),
        col("kafka_timestamp"),
        current_timestamp().alias("ingested_time")
    )


def main():
    # Parse arguments
    args = parse_args()

    spark = (
        SparkSession.builder
            .appName("kafka-to-lakehouse-orders")
            .config("spark.sql.iceberg.check-nullability", "false")
            .getOrCreate()
    )
    spark.sparkContext.setLogLevel("WARN")

    # Setup Lakehouse
    namespace_name = (
        f"`{args.lakehouse_catalog}`."
        f"`{args.lakehouse_namespace}`"
    )

    table_name = (
        f"`{args.lakehouse_catalog}`."
        f"`{args.lakehouse_namespace}`."
        f"`{args.lakehouse_table}`"
    )

    spark.sql(
        f"CREATE NAMESPACE IF NOT EXISTS {namespace_name}"
    )

    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
        order_key BIGINT,
        customer_key BIGINT,
        order_status STRING NOT NULL,
        total_price DECIMAL(15,2),
        order_date DATE NOT NULL,
        order_priority STRING,
        clerk STRING NOT NULL,
        ship_priority INT,
        comment STRING,
        kafka_timestamp TIMESTAMP,
        ingested_time TIMESTAMP
        )
        USING iceberg
        PARTITIONED BY (hours(ingested_time))
        """
    )

    # read orders from Kafka
    orders = read_orders(
        spark,
        args.bootstrap,
        args.kafka_topic,
        args.max_offsets_per_trigger
    )

    # stream order to Lakehouse catalog
    query = (orders.writeStream.format("iceberg").outputMode("append")
             .option("checkpointLocation", args.spark_stream_checkpoint)
             .trigger(processingTime=args.processing_time)
             .queryName("kafka-to-lakehouse-orders").toTable(table_name))
    query.awaitTermination()
    spark.stop()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/kafka_to_spanner.py
#!/usr/bin/env python3

# Requires Jars in Spark env.
#   - spark-cloud-spanner connector
#   - spark-sql-kafka connector
import argparse
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import col, split, to_date, current_timestamp
from pyspark.sql.types import DecimalType, LongType, IntegerType


def parse_args():
    """Parses command-line arguments using argparse."""
    parser = argparse.ArgumentParser(
        description="Stream data from Kafka into a Google Cloud Spanner table."
    )

    parser.add_argument("--bootstrap", required=True, help="Kafka bootstrap servers connection string")
    parser.add_argument("--kafka-topic", required=True, help="Kafka topic name to subscribe to")
    parser.add_argument("--spanner-project", required=True, help="Google Cloud Project ID")
    parser.add_argument("--spanner-instance", required=True, help="Cloud Spanner instance ID")
    parser.add_argument("--spanner-database", required=True, help="Cloud Spanner database name")
    parser.add_argument("--spanner-table", required=True, help="Cloud Spanner table name")
    parser.add_argument("--spark-stream-checkpoint", required=True,
                        help="Remote or local path for streaming checkpoints")
    parser.add_argument(
        "--processing-time",
        default="5 seconds",
        help="Processing time interval for stream trigger (e.g., '10 seconds', '1 minute', '0 seconds'). Default: '5 seconds'"
    )
    parser.add_argument(
        "--max-offsets-per-trigger",
        default="100",
        help="Maximum number of Kafka offsets processed per trigger cycle. Default: 100"
    )

    return parser.parse_args()


def read_orders(spark: SparkSession, bootstrap_servers: str, topic: str, max_offsets_per_trigger: str) -> DataFrame:
    reader = (
        spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", bootstrap_servers)
        .option("subscribe", topic)
        .option("startingOffsets", "earliest")
        .option("maxOffsetsPerTrigger", max_offsets_per_trigger)
        .option("failOnDataLoss", "true")
    )

    # Extract raw value and Kafka record timestamp
    payload = reader.load().select(
        col("value").cast("string").alias("value"),
        col("timestamp").alias("kafka_timestamp")
    )
    fields = split(col("value"), r"\|")
    return payload.select(
        fields.getItem(0).cast(LongType()).alias("order_key"),
        fields.getItem(1).cast(LongType()).alias("customer_key"),
        fields.getItem(2).alias("order_status"),
        fields.getItem(3).cast(DecimalType(15, 2)).alias("total_price"),
        to_date(fields.getItem(4), "yyyy-MM-dd").alias("order_date"),
        fields.getItem(5).alias("order_priority"),
        fields.getItem(6).alias("clerk"),
        fields.getItem(7).cast(IntegerType()).alias("ship_priority"),
        fields.getItem(8).alias("comment"),
        col("kafka_timestamp"),
        current_timestamp().alias("ingested_time")
    )


def write_batch(batch: DataFrame, project: str, instance: str, database: str, table: str):
    if batch.isEmpty():
        return
    (batch.write.format("cloud-spanner")
     .option("projectId", project).option("instanceId", instance)
     .option("databaseId", database).option("table", table)
     .option("mutationType", "insert_or_update")
     .option("assumeIdempotentRows", "true").mode("append").save())


def main():
    # Parse arguments
    args = parse_args()

    spark = SparkSession.builder.appName("kafka-to-spanner-orders").getOrCreate()
    spark.sparkContext.setLogLevel("WARN")

    # read orders from Kafka
    orders = read_orders(
        spark,
        args.bootstrap,
        args.kafka_topic,
        args.max_offsets_per_trigger
    )

    # Stream orders to Spanner using mapped underscore properties
    query = (orders.writeStream.foreachBatch(
        lambda b, _: write_batch(
            b,
            args.spanner_project,
            args.spanner_instance,
            args.spanner_database,
            args.spanner_table
        )
    )
             .option("checkpointLocation", args.spark_stream_checkpoint)
             .trigger(processingTime=args.processing_time)
             .queryName("kafka-to-spanner-orders").start())
    query.awaitTermination()
    spark.stop()


if __name__ == "__main__":
    main()


### 10.2 Upload the PySpark applications

In [ ]:
%%bash
source .env
set -euo pipefail

gcloud storage cp \
  ./src/kafka_to_lakehouse.py \
  ./src/kafka_to_spanner.py \
  "gs://${JOB_BUCKET}/jobs/"


### 10.3 Submit the Kafka-to-Lakehouse job

In [ ]:
%%bash
source .env
set -euo pipefail

KAFKA_PKG="org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3"

LAKEHOUSE_PROPS="^#^\
spark.jars.ivy=/tmp/ivy-lakehouse#\
spark.jars.packages=${KAFKA_PKG}#\
spark.driver.memory=2g#\
spark.sql.defaultCatalog=${LAKEHOUSE_CATALOG}#\
spark.sql.catalog.${LAKEHOUSE_CATALOG}=org.apache.iceberg.spark.SparkCatalog#\
spark.sql.catalog.${LAKEHOUSE_CATALOG}.type=rest#\
spark.sql.catalog.${LAKEHOUSE_CATALOG}.uri=https://biglake.googleapis.com/iceberg/v1/restcatalog#\
spark.sql.catalog.${LAKEHOUSE_CATALOG}.warehouse=gs://${WAREHOUSE_BUCKET}#\
spark.sql.catalog.${LAKEHOUSE_CATALOG}.io-impl=org.apache.iceberg.gcp.gcs.GCSFileIO#\
spark.sql.catalog.${LAKEHOUSE_CATALOG}.header.x-goog-user-project=${PROJECT_ID}#\
spark.sql.catalog.${LAKEHOUSE_CATALOG}.rest.auth.type=google#\
spark.sql.catalog.${LAKEHOUSE_CATALOG}.header.X-Iceberg-Access-Delegation=vended-credentials#\
spark.sql.catalog.${LAKEHOUSE_CATALOG}.rest-metrics-reporting-enabled=false#\
spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"

LAKEHOUSE_JOB_ID="${DEMO_ID}-lakehouse-$(date +%Y%m%d%H%M%S)"

ICEBERG_VERSION="1.10.0"

gcloud dataproc jobs submit pyspark \
  "gs://${JOB_BUCKET}/jobs/kafka_to_lakehouse.py" \
  --id="$LAKEHOUSE_JOB_ID" \
  --cluster="$CLUSTER" \
  --region="$REGION" \
  --properties="$LAKEHOUSE_PROPS" \
  --jars=https://storage-download.googleapis.com/maven-central/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/${ICEBERG_VERSION}/iceberg-spark-runtime-3.5_2.12-${ICEBERG_VERSION}.jar,https://storage-download.googleapis.com/maven-central/maven2/org/apache/iceberg/iceberg-gcp-bundle/${ICEBERG_VERSION}/iceberg-gcp-bundle-${ICEBERG_VERSION}.jar \
  --async \
  -- \
  --bootstrap="$KAFKA_BROKER" \
  --kafka-topic="$KAFKA_TOPIC" \
  --lakehouse-catalog="$LAKEHOUSE_CATALOG" \
  --lakehouse-namespace="$LAKEHOUSE_NAMESPACE" \
  --lakehouse-table="orders" \
  --spark-stream-checkpoint="gs://${JOB_BUCKET}/checkpoints/lakehouse" \
  --processing-time="10 seconds" \
  --max-offsets-per-trigger="100"


### 10.4 Submit the Kafka-to-Spanner job

In [ ]:
%%bash
source .env
set -euo pipefail

KAFKA_PKG="org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3"
SPANNER_PKG="com.google.cloud.spark.spanner:spark-3.5-spanner:1.5.0"

SPANNER_PROPS="^#^\
spark.jars.ivy=/tmp/ivy-spanner#\
spark.jars.packages=${KAFKA_PKG},${SPANNER_PKG}#\
spark.master=local[2]#\
spark.driver.memory=2g"

SPANNER_JOB_ID="${DEMO_ID}-spanner-$(date +%Y%m%d%H%M%S)"

gcloud dataproc jobs submit pyspark \
  "gs://${JOB_BUCKET}/jobs/kafka_to_spanner.py" \
  --id="$SPANNER_JOB_ID" \
  --cluster="$CLUSTER" \
  --region="$REGION" \
  --properties="$SPANNER_PROPS" \
  --async \
  -- \
  --bootstrap="$KAFKA_BROKER" \
  --kafka-topic="$KAFKA_TOPIC" \
  --spanner-project="$PROJECT_ID" \
  --spanner-instance="$SPANNER_INSTANCE" \
  --spanner-database="$SPANNER_DATABASE" \
  --spanner-table="orders" \
  --spark-stream-checkpoint="gs://${JOB_BUCKET}/checkpoints/spanner" \
  --processing-time="5 seconds" \
  --max-offsets-per-trigger="100"


### 10.5 View the jobs in Google Cloud console

In console, go to Managed Spark > Cluster > Jobs to check if the jobs are running.
The jobs would take 1-2 minutes to initialize and be ready to process data.

## 11. Compute aggregates from Lakehouse to Spanner using BigQuery Scheduled & Continuous Queries

This section implements the automated end-to-end aggregation and streaming pipeline into Spanner:

- **5-minute scheduled query (batch precomputation):** scans the Iceberg Lakehouse orders table for affected keys, computes historical `COUNT` and `SUM` snapshots, and appends them to three native BigQuery staging tables (`stg_order_daily`, `stg_clerk_status_daily`, and `stg_order_status_daily`).
- **Three BigQuery Continuous Queries (reverse ETL):** consume changes from the staging tables through `APPENDS()`, calculate averages with `SAFE_DIVIDE`, and export directly into the corresponding Spanner aggregate tables.

The native staging layer is required because BigQuery Continuous Queries cannot read external Iceberg tables directly. https://docs.cloud.google.com/bigquery/docs/continuous-queries-introduction#limitations

### 11.1 Create BigQuery Enterprise reservation and assignment

Continuous Queries require an Enterprise or Enterprise Plus reservation with a `CONTINUOUS` assignment. This demo creates a dedicated 50-slot reservation for the three jobs.

In [ ]:
%%bash
source .env
set -euo pipefail
echo "=== Setting up BigQuery Enterprise Reservation for Continuous Queries ==="

# 1. Create or resize the Enterprise reservation
if ! bq show --reservation --location="$REGION" --project_id="$PROJECT_ID" "$BQ_RESERVATION" &>/dev/null; then
  echo "Creating Enterprise reservation: $BQ_RESERVATION..."
  bq mk --reservation \
    --project_id="$PROJECT_ID" \
    --location="$REGION" \
    --slots=50 \
    --edition=ENTERPRISE \
    "$BQ_RESERVATION"
else
  echo "Updating reservation $BQ_RESERVATION to 50 slots..."
  bq update --reservation \
    --project_id="$PROJECT_ID" \
    --location="$REGION" \
    --slots=50 \
    "$BQ_RESERVATION"
fi

# 2. Create the CONTINUOUS reservation assignment
RAW_ASSIGNMENTS=$(
  bq ls \
    --reservation_assignment \
    --location="$REGION" \
    --project_id="$PROJECT_ID" \
    --format=json 2>/dev/null || true
)

EXISTING_ASSIGNMENT=""
# Only parse with jq if bq returned a JSON array
if [[ "$RAW_ASSIGNMENTS" =~ ^[[:space:]]*\[ ]]; then
  EXISTING_ASSIGNMENT=$(
    echo "$RAW_ASSIGNMENTS" | jq -r --arg assignee "projects/${PROJECT_ID}" '
      .[]
      | select(.jobType == "CONTINUOUS" and .assignee == $assignee)
      | .name // empty
    '
  )
fi

if [ -z "$EXISTING_ASSIGNMENT" ]; then
  echo "Creating reservation assignment for CONTINUOUS jobs..."
  bq mk \
    --project_id="$PROJECT_ID" \
    --location="$REGION" \
    --reservation_assignment \
    --reservation_id="$BQ_RESERVATION" \
    --job_type=CONTINUOUS \
    --assignee_type=PROJECT \
    --assignee_id="$PROJECT_ID"
elif [[ "$EXISTING_ASSIGNMENT" == *"/reservations/${BQ_RESERVATION}/assignments/"* ]]; then
  echo "CONTINUOUS reservation assignment already active: $EXISTING_ASSIGNMENT"
else
  echo "Project $PROJECT_ID already has a CONTINUOUS assignment to another reservation:" >&2
  echo "$EXISTING_ASSIGNMENT" >&2
  exit 1
fi

echo "Enterprise reservation configured for Continuous Queries."

### 11.2 Create native BigQuery staging tables

Create the BigQuery dataset and three native staging tables (`stg_order_daily`, `stg_clerk_status_daily`, and `stg_order_status_daily`) that hold precomputed counts and totals.

In [ ]:
%%bash
source .env
set -euo pipefail

# 1. Create BigQuery dataset if not exists
if ! bq show --dataset "${PROJECT_ID}:${BQ_DATASET}" &>/dev/null; then
  echo "Creating BigQuery dataset: ${BQ_DATASET}..."
  bq mk --dataset --location="$REGION" "${PROJECT_ID}:${BQ_DATASET}"
else
  echo "BigQuery dataset ${BQ_DATASET} already exists."
fi

# 2. Create the 3 native staging tables
echo "Creating staging tables in dataset: ${BQ_DATASET}..."
bq query --project_id="$PROJECT_ID" --location="$REGION" --use_legacy_sql=false "
-- 1. Daily Staging Table
CREATE TABLE IF NOT EXISTS \`${PROJECT_ID}.${BQ_DATASET}.stg_order_daily\` (
  order_date DATE NOT NULL,
  order_count INT64 NOT NULL,
  order_total NUMERIC NOT NULL,
  last_updated TIMESTAMP NOT NULL
)
PARTITION BY DATE(last_updated)
OPTIONS (
  partition_expiration_days = 2
);

-- 2. Clerk & Status Staging Table
CREATE TABLE IF NOT EXISTS \`${PROJECT_ID}.${BQ_DATASET}.stg_clerk_status_daily\` (
  clerk STRING NOT NULL,
  order_status STRING NOT NULL,
  order_date DATE NOT NULL,
  orders_processed INT64 NOT NULL,
  total_revenue NUMERIC NOT NULL,
  last_updated TIMESTAMP NOT NULL
)
PARTITION BY DATE(last_updated)
CLUSTER BY clerk, order_status
OPTIONS (
  partition_expiration_days = 2
);

-- 3. Order Status Staging Table
CREATE TABLE IF NOT EXISTS \`${PROJECT_ID}.${BQ_DATASET}.stg_order_status_daily\` (
  order_status STRING NOT NULL,
  order_date DATE NOT NULL,
  orders_processed INT64 NOT NULL,
  total_revenue NUMERIC NOT NULL,
  last_updated TIMESTAMP NOT NULL
)
PARTITION BY DATE(last_updated)
CLUSTER BY order_status
OPTIONS (
  partition_expiration_days = 2
);
"
echo "Staging tables ready."

### 11.3 Create the precompute scheduled query

The query uses a 5-minute lookback to identify recently affected keys, then recomputes complete `COUNT` and `SUM` snapshots for those keys.

In [ ]:
%%bash
source .env
set -euo pipefail

mkdir -p src/bq_scheduled

cat <<EOF > src/bq_scheduled/precompute_aggregates_to_staging.sql
-- =========================================================================
-- Step 1: Identify dates affected in the last 5-minute schedule interval
-- and pull their historical data for complete cumulative snapshots.
-- =========================================================================
CREATE TEMP TABLE active_orders AS
WITH recent_dates AS (
  SELECT order_date
  FROM `${PROJECT_ID}.${LAKEHOUSE_CATALOG}.${LAKEHOUSE_NAMESPACE}.orders`
  WHERE ingested_time > TIMESTAMP_SUB(@run_time, INTERVAL 5 MINUTE)
    AND ingested_time <= @run_time
)
SELECT
  order_date,
  clerk,
  order_status,
  total_price,
  (ingested_time > TIMESTAMP_SUB(@run_time, INTERVAL 5 MINUTE)
   AND ingested_time <= @run_time) AS is_recent
FROM `${PROJECT_ID}.${LAKEHOUSE_CATALOG}.${LAKEHOUSE_NAMESPACE}.orders`
WHERE order_date IN (SELECT order_date FROM recent_dates);

-- =========================================================================
-- Step 2: Compute all 3 aggregation levels in a single pass.
-- HAVING LOGICAL_OR(is_recent) ensures we only output groups that changed.
-- =========================================================================
CREATE TEMP TABLE aggregate_snapshots AS
SELECT
  order_date,
  clerk,
  order_status,
  COUNT(*) AS orders_count,
  CAST(COALESCE(SUM(total_price), 0) AS NUMERIC) AS total_value,
  GROUPING(clerk) AS is_clerk_aggregated,
  GROUPING(order_status) AS is_status_aggregated,
  @run_time AS last_updated
FROM active_orders
GROUP BY GROUPING SETS (
  (order_date),
  (order_status, order_date),
  (clerk, order_status, order_date)
)
HAVING LOGICAL_OR(is_recent);

-- =========================================================================
-- Step 3: Route into your 3 existing staging tables using is_***_aggregated 
-- conditions.
-- =========================================================================
-- 1. Daily Aggregates
INSERT INTO \`${PROJECT_ID}.${BQ_DATASET}.stg_order_daily\` (
  order_date,
  order_count,
  order_total,
  last_updated
)
SELECT
  order_date,
  orders_count,
  total_value,
  last_updated
FROM aggregate_snapshots
WHERE is_clerk_aggregated = 1 AND is_status_aggregated = 1;

-- 2. Clerk & Status Daily Aggregates
INSERT INTO \`${PROJECT_ID}.${BQ_DATASET}.stg_clerk_status_daily\` (
  clerk,
  order_status,
  order_date,
  orders_processed,
  total_revenue,
  last_updated
)
SELECT
  clerk,
  order_status,
  order_date,
  orders_count,
  total_value,
  last_updated
FROM aggregate_snapshots
WHERE is_clerk_aggregated = 0 AND is_status_aggregated = 0;

-- 3. Order Status Daily Aggregates
INSERT INTO \`${PROJECT_ID}.${BQ_DATASET}.stg_order_status_daily\` (
  order_status,
  order_date,
  orders_processed,
  total_revenue,
  last_updated
)
SELECT
  order_status,
  order_date,
  orders_count,
  total_value,
  last_updated
FROM aggregate_snapshots
WHERE is_clerk_aggregated = 1 AND is_status_aggregated = 0;
EOF

### 11.4 Register the scheduled query

Register the SQL file as a BigQuery scheduled query running every 5 minutes.

In [ ]:
%%bash
source .env
set -euo pipefail

DISPLAY_NAME="precompute_aggregates_to_staging"
PARAMS_JSON=$(jq -n --rawfile q "src/bq_scheduled/${DISPLAY_NAME}.sql" '{"query": $q}')

CONFIG_IDS=$(
  bq ls \
    --project_id="$PROJECT_ID" \
    --transfer_config \
    --transfer_location="$REGION" \
    --format=json 2>/dev/null |
  jq -r --arg display_name "$DISPLAY_NAME" '
    .[]
    | select(.displayName == $display_name)
    | .name // empty
  '
)
while IFS= read -r CONFIG_ID; do
  [ -z "$CONFIG_ID" ] && continue
  echo "Removing existing scheduled query config: $CONFIG_ID"
  bq rm -f --transfer_config "$CONFIG_ID"
done <<< "$CONFIG_IDS"

echo "Creating scheduled query: $DISPLAY_NAME..."
bq mk \
  --transfer_config \
  --project_id="$PROJECT_ID" \
  --data_source=scheduled_query \
  --display_name="$DISPLAY_NAME" \
  --location="$REGION" \
  --schedule="every 5 minutes" \
  --service_account_name="$SPARK_SA" \
  --params="$PARAMS_JSON"

### 11.5 Create the Continuous Query SQL files for Spanner reverse ETL

Create three Continuous Query definitions. Each query reads appends to its staging table, calculates the scalar average, and performs the reverse ETL upsert to its corresponding Spanner table.

Each query replays one day of staging history when it starts. Combined with the two-day staging retention, this lets restarted jobs reprocess recent snapshots instead of creating a gap. Spanner's `change_timestamp_column` resolves replayed snapshots by `last_updated`.

In [ ]:
%%bash
source .env
set -euo pipefail

mkdir -p src/bq_continuous

# ===========================================================================
# 1. Continuous Query -> Spanner: order_daily_aggregate
# ===========================================================================
cat <<EOF > src/bq_continuous/cq_order_daily.sql
EXPORT DATA OPTIONS (
  uri = 'https://spanner.googleapis.com/projects/${PROJECT_ID}/instances/${SPANNER_INSTANCE}/databases/${SPANNER_DATABASE}',
  format = 'CLOUD_SPANNER',
  spanner_options = '{"table": "order_daily_aggregate", "change_timestamp_column": "last_updated"}'
) AS
SELECT
  order_date,
  order_count,
  order_total,
  CAST(SAFE_DIVIDE(order_total, order_count) AS NUMERIC) AS order_total_average,
  last_updated
FROM APPENDS(
  TABLE \`${PROJECT_ID}.${BQ_DATASET}.stg_order_daily\`,
  CURRENT_TIMESTAMP() - INTERVAL 1 DAY
);
EOF

# ===========================================================================
# 2. Continuous Query -> Spanner: clerk_status_daily_aggregate
# ===========================================================================
cat <<EOF > src/bq_continuous/cq_clerk_status_daily.sql
EXPORT DATA OPTIONS (
  uri = 'https://spanner.googleapis.com/projects/${PROJECT_ID}/instances/${SPANNER_INSTANCE}/databases/${SPANNER_DATABASE}',
  format = 'CLOUD_SPANNER',
  spanner_options = '{"table": "clerk_status_daily_aggregate", "change_timestamp_column": "last_updated"}'
) AS
SELECT
  clerk,
  order_status,
  order_date,
  orders_processed,
  total_revenue,
  CAST(SAFE_DIVIDE(total_revenue, orders_processed) AS NUMERIC) AS avg_order_value,
  last_updated
FROM APPENDS(
  TABLE \`${PROJECT_ID}.${BQ_DATASET}.stg_clerk_status_daily\`,
  CURRENT_TIMESTAMP() - INTERVAL 1 DAY
);
EOF

# ===========================================================================
# 3. Continuous Query -> Spanner: order_status_daily_aggregate
# ===========================================================================
cat <<EOF > src/bq_continuous/cq_order_status_daily.sql
EXPORT DATA OPTIONS (
  uri = 'https://spanner.googleapis.com/projects/${PROJECT_ID}/instances/${SPANNER_INSTANCE}/databases/${SPANNER_DATABASE}',
  format = 'CLOUD_SPANNER',
  spanner_options = '{"table": "order_status_daily_aggregate", "change_timestamp_column": "last_updated"}'
) AS
SELECT
  order_status,
  order_date,
  orders_processed,
  total_revenue,
  CAST(SAFE_DIVIDE(total_revenue, orders_processed) AS NUMERIC) AS avg_order_value,
  last_updated
FROM APPENDS(
  TABLE \`${PROJECT_ID}.${BQ_DATASET}.stg_order_status_daily\`,
  CURRENT_TIMESTAMP() - INTERVAL 1 DAY
);
EOF

echo "Continuous query SQL definitions created in src/bq_continuous/."

### 11.6 Start the Continuous Queries

Submit the three Continuous Queries in the background with `--continuous=true` and `--sync=false`. The launch cell first cancels running jobs created by an earlier execution, preventing duplicate consumers when the cell is rerun.

Continuous Queries started with a service account have a maximum runtime of 150 days. Rerun this cell before or after that limit, and after any extended interruption. The one-day replay makes a restart safe as long as the required staging partitions have not expired.

In [ ]:
%%bash
source .env
set -euo pipefail

RUNNING_CQ_JOBS=$(
  bq query \
    --project_id="$PROJECT_ID" \
    --location="$REGION" \
    --use_legacy_sql=false \
    --format=csv \
    --quiet \
    "
      SELECT job_id
      FROM \`region-${REGION}\`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
      WHERE continuous IS TRUE
        AND state = 'RUNNING'
        AND (
          STARTS_WITH(job_id, 'cq_order_daily_')
          OR STARTS_WITH(job_id, 'cq_clerk_status_daily_')
          OR STARTS_WITH(job_id, 'cq_order_status_daily_')
        )
    " |
  tail -n +2
)

while IFS= read -r JOB_ID; do
  [ -z "$JOB_ID" ] && continue
  echo "Cancelling existing Continuous Query: $JOB_ID..."
  bq --project_id="$PROJECT_ID" --location="$REGION" cancel "$JOB_ID"
done <<< "$RUNNING_CQ_JOBS"

# Generate a timestamp suffix to guarantee uniqueness across runs
RUN_TS=$(date +%Y%m%d_%H%M%S)

echo "=== Launching BigQuery Continuous Queries with Service Account ($SPARK_SA) ==="

# 1. Order Daily Aggregate
echo "Starting Continuous Query: order_daily_aggregate..."
bq query \
  --project_id="$PROJECT_ID" \
  --location="$REGION" \
  --job_id="cq_order_daily_${RUN_TS}" \
  --label="demo:sparkdataboost" \
  --use_legacy_sql=false \
  --continuous=true \
  --sync=false \
  --connection_property=service_account="$SPARK_SA" \
  < src/bq_continuous/cq_order_daily.sql

# 2. Clerk & Status Daily Aggregate
echo "Starting Continuous Query: clerk_status_daily_aggregate..."
bq query \
  --project_id="$PROJECT_ID" \
  --location="$REGION" \
  --job_id="cq_clerk_status_daily_${RUN_TS}" \
  --label="demo:sparkdataboost" \
  --use_legacy_sql=false \
  --continuous=true \
  --sync=false \
  --connection_property=service_account="$SPARK_SA" \
  < src/bq_continuous/cq_clerk_status_daily.sql

# 3. Order Status Daily Aggregate
echo "Starting Continuous Query: order_status_daily_aggregate..."
bq query \
  --project_id="$PROJECT_ID" \
  --location="$REGION" \
  --job_id="cq_order_status_daily_${RUN_TS}" \
  --label="demo:sparkdataboost" \
  --use_legacy_sql=false \
  --continuous=true \
  --sync=false \
  --connection_property=service_account="$SPARK_SA" \
  < src/bq_continuous/cq_order_status_daily.sql

echo "All 3 Continuous Queries successfully started in the background under $SPARK_SA."

## 12. Produce and verify demo data

### 12.1 Run the producer

The data flow is now ready to process orders. We can start the Kafka producer.

In [ ]:
%%bash
source .env
set -euo pipefail

gcloud compute ssh "$KAFKA_VM" \
  --zone="$ZONE" \
  --tunnel-through-iap \
  --command='
    set -euo pipefail

    source ~/.env

    docker run --rm \
      --name tpch-generator \
      --network host \
      tpch-generator:local \
      --local \
      --bootstrap-servers="${KAFKA_BROKER}" \
      --max-messages=1000 \
      --target-throughput=100 \
      --topic="${KAFKA_TOPIC}" \
      --date-from="1 month ago" \
      --date-to="today"
  '


With the settings above, the producer will send 1000 messages at a rate of 100 messages per second, which will take about 10 seconds to complete.

In GCP console, you can check that Lakehouse and Spanner tables are being populated.


### 12.2 Verify Spanner aggregate table population

Check Spanner after the scheduled query has produced staging snapshots and the Continuous Queries have exported them.

> The scheduled query runs every five minutes. Allow at least one complete schedule interval before expecting aggregate rows.

In [ ]:
%%bash
source .env
set -euo pipefail

echo "=== Spanner Aggregate Tables Record Counts ==="
for TABLE in order_daily_aggregate clerk_status_daily_aggregate order_status_daily_aggregate; do
  echo "Table: $TABLE"
  gcloud spanner databases execute-sql "$SPANNER_DATABASE" \
    --instance="$SPANNER_INSTANCE" \
    --sql="SELECT COUNT(*) AS orders_count, MAX(last_updated) AS latest_sync FROM $TABLE;"
done


## 13. Visualize the data in JupyterLab Notebook

In your Managed Spark cluster, navigate to Web Interfaces.

![spark-webinterfaces.png](imgs/spark-webinterfaces.png)

Open a **JupyterLab** notebook.

From the Launcher, create a new **Python 3** notebook. (Note: Select Python 3, not PySpark.)

> **Run the next two code cells there, not on your local kernel.** They need
> the cluster's private-network access to Spanner and rely on packages
> (`pyspark`, `matplotlib`, `seaborn`) preinstalled on the cluster's Jupyter
> image.

### Part 1: Dashboard querying pre-aggregated tables

Paste this cell into the cluster's Python 3 notebook to query the
pre-computed aggregate tables from the Lakehouse path.


In [ ]:
# =========================================================================
# SPANNER ORDERS DASHBOARD
# Rolling one-month reporting period
#
# Dashboard panels:
#   1. Daily total revenue
#   2A. Daily revenue by order status
#   2B. Daily revenue percentage by order status
#   3. Top 10 clerks by order count across all statuses
#   4. Top 10 clerks by total revenue across all statuses
# =========================================================================

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

PROJECT_ID = "your-project-id"
SPANNER_INSTANCE = "spark-databoost-demo-instance"
SPANNER_DATABASE = "demo"

if PROJECT_ID == "your-project-id":
    raise ValueError("Set PROJECT_ID to the project used in section 1")


# -------------------------------------------------------------------------
# 1. INITIALIZE SPARK SESSION WITH THE SPANNER CONNECTOR
# -------------------------------------------------------------------------

spark = (
    SparkSession.builder
    .appName("Spanner Aggregates Dashboard")
    .config(
        "spark.jars",
        "gs://spark-lib/spanner/spark-3.5-spanner-1.5.0.jar"
    )
    .getOrCreate()
)


# -------------------------------------------------------------------------
# 2. DEFINE THE ROLLING ONE-MONTH REPORTING PERIOD
# -------------------------------------------------------------------------

# Inclusive date range:
# start_date = one calendar month before today
# end_date   = today

start_date = F.add_months(F.current_date(), -1)
end_date = F.current_date()


# -------------------------------------------------------------------------
# 3. READ AND FILTER THE PRE-AGGREGATED SPANNER TABLES
# -------------------------------------------------------------------------

df_day_agg = (
    spark.read
    .format("cloud-spanner")
    .option("projectId", PROJECT_ID)
    .option("instanceId", SPANNER_INSTANCE)
    .option("databaseId", SPANNER_DATABASE)
    .option("table", "order_daily_aggregate")
    .load()
    .filter(
        F.to_date(F.col("order_date")).between(start_date, end_date)
    )
)


df_status_agg = (
    spark.read
    .format("cloud-spanner")
    .option("projectId", PROJECT_ID)
    .option("instanceId", SPANNER_INSTANCE)
    .option("databaseId", SPANNER_DATABASE)
    .option("table", "order_status_daily_aggregate")
    .load()
    .filter(
        F.to_date(F.col("order_date")).between(start_date, end_date)
    )
)


df_clerk_status_agg = (
    spark.read
    .format("cloud-spanner")
    .option("projectId", PROJECT_ID)
    .option("instanceId", SPANNER_INSTANCE)
    .option("databaseId", SPANNER_DATABASE)
    .option("table", "clerk_status_daily_aggregate")
    .load()
    .filter(
        F.to_date(F.col("order_date")).between(start_date, end_date)
    )
)


# -------------------------------------------------------------------------
# 4. CONVERT THE SPARK DATAFRAMES TO PANDAS DATAFRAMES
# -------------------------------------------------------------------------

pdf_day = df_day_agg.orderBy("order_date").toPandas()
pdf_status = df_status_agg.orderBy("order_date").toPandas()
pdf_clerk = df_clerk_status_agg.orderBy("order_date").toPandas()


# -------------------------------------------------------------------------
# 5. PREPARE THE PANDAS DATAFRAMES
# -------------------------------------------------------------------------

for pdf in [pdf_day, pdf_status, pdf_clerk]:
    pdf["order_date"] = pd.to_datetime(
        pdf["order_date"],
        errors="coerce"
    )


# Convert Spanner NUMERIC and INT64 values into Pandas numeric types.

pdf_day["order_total"] = pd.to_numeric(
    pdf_day["order_total"],
    errors="coerce"
).fillna(0.0)

pdf_status["total_revenue"] = pd.to_numeric(
    pdf_status["total_revenue"],
    errors="coerce"
).fillna(0.0)

pdf_clerk["total_revenue"] = pd.to_numeric(
    pdf_clerk["total_revenue"],
    errors="coerce"
).fillna(0.0)

pdf_clerk["orders_processed"] = pd.to_numeric(
    pdf_clerk["orders_processed"],
    errors="coerce"
).fillna(0)


# Make order-status labels more descriptive.

status_labels = {
    "F": "F - Fulfilled",
    "O": "O - Open",
    "P": "P - Processing"
}

pdf_status["order_status_label"] = (
    pdf_status["order_status"]
    .map(status_labels)
    .fillna(pdf_status["order_status"])
)

pdf_clerk["order_status_label"] = (
    pdf_clerk["order_status"]
    .map(status_labels)
    .fillna(pdf_clerk["order_status"])
)


# -------------------------------------------------------------------------
# 6. CREATE THE ORDER-STATUS PIVOT TABLES
# -------------------------------------------------------------------------

pivot_status_abs = (
    pdf_status
    .pivot_table(
        index="order_date",
        columns="order_status_label",
        values="total_revenue",
        aggfunc="sum",
        fill_value=0
    )
    .sort_index()
)

pivot_status_pct = (
    pivot_status_abs
    .div(
        pivot_status_abs.sum(axis=1).replace(0, pd.NA),
        axis=0
    )
    .mul(100)
    .fillna(0)
)


# -------------------------------------------------------------------------
# 7. GENERATE THE ACTUAL REPORTING-PERIOD LABEL
# -------------------------------------------------------------------------

all_dates = pd.concat(
    [
        pdf_day["order_date"],
        pdf_status["order_date"],
        pdf_clerk["order_date"]
    ],
    ignore_index=True
).dropna()

if not all_dates.empty:
    period_start_label = all_dates.min().strftime("%b %d, %Y")
    period_end_label = all_dates.max().strftime("%b %d, %Y")
    period_label = f"{period_start_label} to {period_end_label}"
else:
    period_label = "Rolling One-Month Period"


# -------------------------------------------------------------------------
# 8. CALCULATE THE TOP 10 CLERKS BY ORDER COUNT
# -------------------------------------------------------------------------

# The aggregate table has one row per clerk, status, and date.
# Summing only by clerk combines all statuses and dates in the period.

top_clerks_by_count = (
    pdf_clerk
    .groupby("clerk", as_index=False)
    .agg(order_count=("orders_processed", "sum"))
    .nlargest(10, "order_count")
    .sort_values("order_count", ascending=False)
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------
# 9. CALCULATE THE TOP 10 CLERKS BY TOTAL REVENUE
# -------------------------------------------------------------------------

# This ranking is independent of the order-count ranking.

top_clerks_by_revenue = (
    pdf_clerk
    .groupby("clerk", as_index=False)
    .agg(total_revenue=("total_revenue", "sum"))
    .nlargest(10, "total_revenue")
    .sort_values("total_revenue", ascending=False)
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------
# 10. BUILD THE FIVE-PANEL DASHBOARD
# -------------------------------------------------------------------------

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(
    nrows=5,
    ncols=1,
    figsize=(14, 29),
    gridspec_kw={
        "height_ratios": [1, 1.25, 1.25, 1.6, 1.6]
    }
)

fig.suptitle(
    (
        "Spanner Orders Dashboard\n"
        f"Rolling One-Month Period: {period_label}"
    ),
    fontsize=17,
    fontweight="bold",
    y=0.995
)


# -------------------------------------------------------------------------
# GRAPH 1: TOTAL DAILY REVENUE TREND
# -------------------------------------------------------------------------

sns.lineplot(
    ax=axes[0],
    data=pdf_day,
    x="order_date",
    y="order_total",
    marker="o",
    color="navy",
    linewidth=2.5
)

axes[0].set_title(
    f"1. Daily Total Revenue | {period_label}",
    fontsize=13,
    fontweight="bold"
)
axes[0].set_xlabel("Order Date")
axes[0].set_ylabel("Revenue ($)")
axes[0].tick_params(axis="x", rotation=30)


# -------------------------------------------------------------------------
# GRAPH 2A: DAILY REVENUE BREAKDOWN BY ORDER STATUS
# -------------------------------------------------------------------------

pivot_status_abs.plot(
    kind="bar",
    stacked=True,
    ax=axes[1],
    colormap="Set2",
    edgecolor="black",
    linewidth=0.8,
    alpha=0.85
)

axes[1].set_title(
    f"2A. Daily Revenue by Order Status | {period_label}",
    fontsize=13,
    fontweight="bold"
)
axes[1].set_ylabel("Total Revenue ($)")
axes[1].set_xlabel("Order Date")
axes[1].legend(title="Order Status", loc="upper left")
axes[1].set_xticklabels(
    [date.strftime("%b %d") for date in pivot_status_abs.index],
    rotation=30,
    ha="right"
)


# -------------------------------------------------------------------------
# GRAPH 2B: DAILY REVENUE DISTRIBUTION BY ORDER STATUS
# -------------------------------------------------------------------------

pivot_status_pct.plot(
    kind="bar",
    stacked=True,
    ax=axes[2],
    colormap="Set2",
    edgecolor="black",
    linewidth=0.8,
    alpha=0.85
)

axes[2].set_title(
    f"2B. Daily Revenue Distribution by Order Status | {period_label}",
    fontsize=13,
    fontweight="bold"
)
axes[2].set_ylabel("Percentage (%)")
axes[2].set_xlabel("Order Date")
axes[2].set_ylim(0, 100)
axes[2].legend(title="Order Status", loc="upper left")
axes[2].set_xticklabels(
    [date.strftime("%b %d") for date in pivot_status_pct.index],
    rotation=30,
    ha="right"
)


# -------------------------------------------------------------------------
# GRAPH 3: TOP 10 CLERKS BY ORDER COUNT, ALL STATUSES
# -------------------------------------------------------------------------

if (
        not top_clerks_by_count.empty
        and top_clerks_by_count["order_count"].sum() > 0
):
    count_pie_labels = (
        top_clerks_by_count["clerk"]
        .astype(str)
        .str.replace(r"^Clerk#0*", "Clerk#", regex=True)
    )

    top_10_total_orders = top_clerks_by_count["order_count"].sum()

    def format_order_pie_label(percent):
        count = int(round(percent * top_10_total_orders / 100))
        return f"{percent:.1f}%\n{count:,} orders"

    count_pie_colors = sns.color_palette(
        "Set2",
        n_colors=len(top_clerks_by_count)
    )

    count_wedges, count_label_texts, count_value_texts = axes[3].pie(
        top_clerks_by_count["order_count"],
        labels=count_pie_labels,
        autopct=format_order_pie_label,
        startangle=90,
        counterclock=False,
        colors=count_pie_colors,
        pctdistance=0.72,
        labeldistance=1.08,
        wedgeprops={
            "edgecolor": "white",
            "linewidth": 1.5
        },
        textprops={
            "fontsize": 9
        }
    )

    for value_text in count_value_texts:
        value_text.set_fontsize(8)
        value_text.set_fontweight("bold")

    axes[3].set_title(
        f"3. Top 10 Clerks by Order Count, All Statuses | {period_label}",
        fontsize=13,
        fontweight="bold",
        pad=18
    )
    axes[3].axis("equal")
    axes[3].legend(
        count_wedges,
        [
            f"{clerk}: {int(count):,} orders"
            for clerk, count in zip(
            count_pie_labels,
            top_clerks_by_count["order_count"]
        )
        ],
        title=(
            "Clerk and Order Count\n"
            f"Top 10 Total: {int(top_10_total_orders):,}"
        ),
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        fontsize=9
    )
else:
    axes[3].text(
        0.5,
        0.5,
        "No clerk order data available\nfor the selected reporting period",
        horizontalalignment="center",
        verticalalignment="center",
        transform=axes[3].transAxes,
        fontsize=12
    )
    axes[3].set_title(
        f"3. Top 10 Clerks by Order Count, All Statuses | {period_label}",
        fontsize=13,
        fontweight="bold"
    )
    axes[3].axis("off")


# -------------------------------------------------------------------------
# GRAPH 4: TOP 10 CLERKS BY TOTAL REVENUE, ALL STATUSES
# -------------------------------------------------------------------------

if (
        not top_clerks_by_revenue.empty
        and top_clerks_by_revenue["total_revenue"].sum() > 0
):
    revenue_pie_labels = (
        top_clerks_by_revenue["clerk"]
        .astype(str)
        .str.replace(r"^Clerk#0*", "Clerk#", regex=True)
    )

    top_10_total_revenue = top_clerks_by_revenue["total_revenue"].sum()

    def format_revenue_pie_label(percent):
        return f"{percent:.1f}%"

    revenue_pie_colors = sns.color_palette(
        "Set3",
        n_colors=len(top_clerks_by_revenue)
    )

    revenue_wedges, revenue_label_texts, revenue_value_texts = axes[4].pie(
        top_clerks_by_revenue["total_revenue"],
        labels=revenue_pie_labels,
        autopct=format_revenue_pie_label,
        startangle=90,
        counterclock=False,
        colors=revenue_pie_colors,
        pctdistance=0.72,
        labeldistance=1.08,
        wedgeprops={
            "edgecolor": "white",
            "linewidth": 1.5
        },
        textprops={
            "fontsize": 9
        }
    )

    for value_text in revenue_value_texts:
        value_text.set_fontsize(9)
        value_text.set_fontweight("bold")

    axes[4].set_title(
        f"4. Top 10 Clerks by Total Order Revenue, All Statuses | {period_label}",
        fontsize=13,
        fontweight="bold",
        pad=18
    )
    axes[4].axis("equal")
    axes[4].legend(
        revenue_wedges,
        [
            f"{clerk}: ${revenue:,.2f}"
            for clerk, revenue in zip(
            revenue_pie_labels,
            top_clerks_by_revenue["total_revenue"]
        )
        ],
        title=(
            "Clerk and Total Revenue\n"
            f"Top 10 Total: ${top_10_total_revenue:,.2f}"
        ),
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        fontsize=9
    )
else:
    axes[4].text(
        0.5,
        0.5,
        "No clerk revenue data available\nfor the selected reporting period",
        horizontalalignment="center",
        verticalalignment="center",
        transform=axes[4].transAxes,
        fontsize=12
    )
    axes[4].set_title(
        f"4. Top 10 Clerks by Total Order Revenue, All Statuses | {period_label}",
        fontsize=13,
        fontweight="bold"
    )
    axes[4].axis("off")


# -------------------------------------------------------------------------
# 11. FINALIZE AND DISPLAY THE DASHBOARD
# -------------------------------------------------------------------------

# Reserve space on the right for both pie-chart legends.
plt.tight_layout(rect=[0, 0, 0.82, 0.985])
plt.show()


### Part 2: Dashboard with ad-hoc queries using Data Boost

Using Spanner Data Boost, we can run ad-hoc queries on the raw `orders`
table. Paste this cell into a second **Python 3** notebook on the cluster.

Note:
- Data Boost is enabled via `.option("enableDataBoost", "true")`.
- To force using the Columnar scan we are using Spanner `@{scan_method=columnar}` hint in the SQL query sent to Spanner.
- Sending a direct query `.option("query", query)` is introduced by spark-3.5-spanner-1.5.0 (version >= 1.5.0).


In [ ]:
from datetime import date, timedelta
from IPython.display import display, Markdown
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

PROJECT_ID = "your-project-id"
SPANNER_INSTANCE = "spark-databoost-demo-instance"
SPANNER_DATABASE = "demo"

if PROJECT_ID == "your-project-id":
    raise ValueError("Set PROJECT_ID to the project used in section 1")

spark = (
    SparkSession.builder
    .appName("Spanner Aggregates Data Boost Ad-hoc Queries - Direct Query")
    .config(
        "spark.jars",
        "gs://spark-lib/spanner/spark-3.5-spanner-1.5.0.jar"
    )
    .getOrCreate()
)

cutoff_date = (date.today() - timedelta(days=3)).isoformat()

# Force columnar scan via the @{scan_method=columnar} hint
query = f"""
@{{scan_method=columnar}}
SELECT 
    order_key,
    customer_key,
    order_status,
    total_price,
    order_date,
    order_priority,
    clerk
FROM orders
WHERE order_status != 'F'
  AND order_date < DATE '{cutoff_date}'
"""

# Read using direct query and Data Boost
unfulfilled_orders_raw = (
    spark.read
    .format("cloud-spanner")
    .option("projectId", PROJECT_ID)
    .option("instanceId", SPANNER_INSTANCE)
    .option("databaseId", SPANNER_DATABASE)
    .option("query", query)
    .option("enableDataBoost", "true")
    .load()
)

# Order in Spark
unfulfilled_orders = unfulfilled_orders_raw.orderBy(F.col("order_date").asc())

display(Markdown("### Spark Physical Plan"))
unfulfilled_orders.explain("formatted")

display(Markdown("### ⚠️ Unfulfilled Orders (> 3 Days Old)"))
unfulfilled_orders.show(truncate=False)

## 14. Cleanup

### 14.1 Clean-up Spark resources

In [ ]:
%%bash
source .env
set -euo pipefail

# 1. Stop Spark Dataproc streaming jobs
echo "=== Stopping Spark Dataproc Jobs ==="
for JOB_ID in $(
  gcloud dataproc jobs list \
    --region="$REGION" \
    --cluster="$CLUSTER" \
    --filter='status.state=RUNNING OR status.state=PENDING OR status.state=SETUP_DONE' \
    --format='value(reference.jobId)' 2>/dev/null
); do
  echo "Killing Dataproc job: $JOB_ID..."
  gcloud dataproc jobs kill "$JOB_ID" --region="$REGION" --quiet || true
done

# 2. Delete the Spark cluster
gcloud dataproc clusters delete "$CLUSTER" \
  --region="$REGION" \
  --quiet || true


### 14.2 Clean-up BigQuery resources

In [ ]:
%%bash
source .env
set -euo pipefail

echo "=== Cleaning up BigQuery Resources ==="

# 1. Cancel running BigQuery Continuous Queries
echo "=== Stopping demo Continuous Queries ==="

for JOB_ID in $(
  bq query --project_id="$PROJECT_ID" --location="$REGION" --use_legacy_sql=false --format=csv --quiet "
    SELECT job_id
    FROM \`region-${REGION}\`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
    WHERE continuous IS TRUE
      AND state = 'RUNNING'
      AND (
        STARTS_WITH(job_id, 'cq_order_daily_')
        OR STARTS_WITH(job_id, 'cq_clerk_status_daily_')
        OR STARTS_WITH(job_id, 'cq_order_status_daily_')
      )
  " 2>/dev/null | tail -n +2
); do
  [[ -z "$JOB_ID" ]] && continue
  echo "Cancelling Continuous Query: $JOB_ID..."
  bq --project_id="$PROJECT_ID" --location="$REGION" cancel "$JOB_ID" || true
done

# 2. Delete the scheduled query
DISPLAY_NAME="precompute_aggregates_to_staging"
while IFS= read -r CONFIG_ID; do
  [[ -z "$CONFIG_ID" ]] && continue
  echo "Deleting scheduled query: $DISPLAY_NAME ($CONFIG_ID)..."
  bq rm -f --transfer_config "$CONFIG_ID" || true
done < <(
  bq ls \
    --project_id="$PROJECT_ID" \
    --transfer_config \
    --transfer_location="$REGION" \
    --format=json 2>/dev/null |
    jq -r --arg display_name "$DISPLAY_NAME" '
      .[]
      | select(.displayName == $display_name)
      | .name // empty
    '
)

# 3. Delete intermediate staging dataset and tables
echo "Deleting BigQuery staging dataset: ${BQ_DATASET}..."
bq rm -r -f -d "${PROJECT_ID}:${BQ_DATASET}" || true

# 4. Delete CONTINUOUS reservation assignments
echo "=== Deleting BigQuery Reservation Assignments ==="
ASSIGNMENT_NAMES=$(
  bq ls \
    --reservation_assignment \
    --location="$REGION" \
    --project_id="$PROJECT_ID" \
    --format=prettyjson 2>/dev/null |
  jq -r \
    --arg reservation "$BQ_RESERVATION" \
    '
      .[]
      | select(
          .name
          | contains("/reservations/" + $reservation + "/assignments/")
        )
      | .name
    '
)

if [ -z "$ASSIGNMENT_NAMES" ]; then
  echo "No assignments found for reservation: $BQ_RESERVATION"
else
  while IFS= read -r ASSIGNMENT_NAME; do
    [[ -z "$ASSIGNMENT_NAME" ]] && continue
    ASSIGNMENT_ID="${ASSIGNMENT_NAME##*/}"
    RESERVATION_PATH="${ASSIGNMENT_NAME%/assignments/*}"
    RESERVATION_ID="${RESERVATION_PATH##*/}"
    ASSIGNMENT_REF="${RESERVATION_ID}.${ASSIGNMENT_ID}"

    echo "Deleting assignment: $ASSIGNMENT_REF"
    bq rm \
      --force \
      --reservation_assignment \
      --location="$REGION" \
      --project_id="$PROJECT_ID" \
      "$ASSIGNMENT_REF"
  done <<< "$ASSIGNMENT_NAMES"
fi

# 5. Delete the reservation
if bq show --reservation --location="$REGION" --project_id="$PROJECT_ID" "$BQ_RESERVATION" &>/dev/null; then
  echo "Deleting reservation: $BQ_RESERVATION..."
  bq rm -f --reservation --location="$REGION" --project_id="$PROJECT_ID" "$BQ_RESERVATION" || true
fi

echo "All BigQuery resources cleaned up."

### 14.3 Delete compute and database resources

In [ ]:
%%bash
source .env
set -euo pipefail

gcloud spanner instances delete "$SPANNER_INSTANCE" \
  --quiet || true

gcloud compute instances delete "$KAFKA_VM" \
  --zone="$ZONE" \
  --quiet || true


### 14.4 Delete the Lakehouse catalog

In the Google Cloud console:

1. Open **BigQuery / Lakehouse**.
2. Select the Iceberg REST catalog.
3. Delete the catalog.


### 14.5 Delete the storage buckets

In [ ]:
%%bash
source .env
set -euo pipefail

gcloud storage rm \
  --recursive \
  "gs://${JOB_BUCKET}" || true

gcloud storage rm \
  --recursive \
  "gs://${WAREHOUSE_BUCKET}" || true


### 14.6 Delete the network resources

In [ ]:
%%bash
source .env
set -euo pipefail

gcloud compute routers nats delete "$NAT" \
  --router="$ROUTER" \
  --region="$REGION" \
  --quiet || true

gcloud compute routers delete "$ROUTER" \
  --region="$REGION" \
  --quiet || true

gcloud compute firewall-rules delete \
  "allow-${NETWORK}-internal" \
  "allow-${NETWORK}-ssh-iap" \
  --quiet || true

gcloud compute networks subnets delete "$SUBNET" \
  --region="$REGION" \
  --quiet || true
gcloud compute networks delete "$NETWORK" \
  --quiet || true


### 14.7 Delete the Service Account

In [ ]:
%%bash
source .env
set -euo pipefail

gcloud iam service-accounts delete "$SPARK_SA" \
  --quiet || true
